In [ ]:
!pip install torch torchvision transformers pillow tqdm lightgbm scikit-learn pandas numpy sentence-transformers termcolor accelerate --quiet

In [ ]:
import pandas as pd
import numpy as np
import torch 
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import CLIPProcessor, CLIPModel, DistilBertTokenizer, DistilBertModel
from PIL import Image
import requests
from io import BytesIO
from tqdm import tqdm
import warnings
import os
import re
from termcolor import colored
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import lightgbm as lgb

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class Config:
    TRAIN_FILE = "dataset/train.csv"
    TEST_FILE = "dataset/test.csv"
    TRAIN_EMBEDDINGS_FILE = "embeddings/train_embeddings.npz"
    TEST_EMBEDDINGS_FILE = "embeddings/test_embeddings.npz"
    
    CLIP_MODEL = 'openai/clip-vit-base-patch32'
    TEXT_MODEL = 'distilbert-base-uncased'
    MAX_TEXT_LENGTH = 256
    BATCH_SIZE = 32
    EPOCHS = 5
    LEARNING_RATE = 2e-5
    VAL_SPLIT_SIZE = 0.1
    
    USE_SVD = True
    N_COMPONENTS = 256
    
    LGB_PARAMS = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': 8,
        'min_child_samples': 20,
        'subsample': 0.8,
        'subsample_freq': 1,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

In [ ]:
train_df = pd.read_csv(Config.TRAIN_FILE)
test_df = pd.read_csv(Config.TEST_FILE)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nPrice statistics:")
print(train_df['price'].describe())

cap_value = train_df['price'].quantile(0.95)
train_df['price'] = np.minimum(train_df['price'], cap_value)

print(colored(f"\n95th percentile price cap applied: {cap_value:.2f}", 'cyan'))

Train shape: (75000, 4)
Test shape: (75000, 3)

Price statistics:
count    75000.000000
mean        23.647654
std         33.376932
min          0.130000
25%          6.795000
50%         14.000000
75%         28.625000
max       2796.000000
Name: price, dtype: float64

95th percentile price cap applied: 75.71


In [ ]:
def extract_features_from_text(text):
    features = {}

    quantity_patterns = [
        r'(\d+)\s*(?:Pack|pack|Count|count)',
        r'IPQ[:\s]*(\d+)',
        r'(\d+)\s*(?:Ounce|ounce|oz|Oz)',
        r'(\d+\.?\d*)\s*(?:Pound|pound|lb|Lb)'
    ]
    quantities = []
    for pattern in quantity_patterns:
        matches = re.findall(pattern, str(text))
        quantities.extend([float(m) for m in matches])
    features['quantity'] = max(quantities) if quantities else 1.0

    weight_patterns = [
        r'(\d+\.?\d*)\s*(?:oz|ounce)',
        r'(\d+\.?\d*)\s*(?:lb|pound)',
        r'(\d+\.?\d*)\s*(?:ml|milliliter)',
        r'(\d+\.?\d*)\s*(?:l|liter)'
    ]
    weights = []
    for pattern in weight_patterns:
        matches = re.findall(pattern, str(text).lower())
        weights.extend([float(m) for m in matches])
    features['weight'] = max(weights) if weights else 0.0

    features['text_length'] = len(str(text))
    features['word_count'] = len(str(text).split())
    features['capital_ratio'] = sum(1 for c in str(text) if c.isupper()) / len(str(text)) if len(str(text)) > 0 else 0
    features['digit_ratio'] = sum(1 for c in str(text) if c.isdigit()) / len(str(text)) if len(str(text)) > 0 else 0

    return features

train_text_features = train_df['catalog_content'].apply(extract_features_from_text)
test_text_features = test_df['catalog_content'].apply(extract_features_from_text)

train_text_df = pd.DataFrame(train_text_features.tolist())
test_text_df = pd.DataFrame(test_text_features.tolist())

print(f"Extracted features: {train_text_df.columns.tolist()}")
print(train_text_df.head())

Extracted features: ['quantity', 'weight', 'text_length', 'word_count', 'capital_ratio', 'digit_ratio']
   quantity  weight  text_length  word_count  capital_ratio  digit_ratio
0      12.0   12.00           91          18       0.153846     0.065934
1       8.0    8.00          511          80       0.076321     0.033268
2       9.0    1.90          328          59       0.088415     0.036585
3      25.0   11.25         1318         211       0.045524     0.012898
4       7.0   12.70          155          28       0.096774     0.083871


In [ ]:
class CLIPFeatureExtractor:

    def __init__(self, model_name=Config.CLIP_MODEL, batch_size=Config.BATCH_SIZE):
      
        self.device = device
        print(f"Loading CLIP model: {model_name}")

        self.model = CLIPModel.from_pretrained(model_name).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.model.eval()
        self.batch_size = batch_size

        self.image_embed_dim = self.model.config.projection_dim
        self.text_embed_dim = self.model.config.projection_dim

        print(f"Model loaded successfully!")
        print(f"Embedding dimension: {self.image_embed_dim}")

    def get_image_embeddings(self, image_urls, max_retries=3):
        embeddings = []

        for i in tqdm(range(0, len(image_urls), self.batch_size), desc="Processing images"):
            batch_urls = image_urls[i:i+self.batch_size]
            batch_images = []

            for url in batch_urls:
                img = None
                for attempt in range(max_retries):
                    try:
                        response = requests.get(url, timeout=5)
                        img = Image.open(BytesIO(response.content)).convert('RGB')
                        break
                    except:
                        if attempt < max_retries - 1:
                            continue
                        else:
                            img = Image.new('RGB', (224, 224), color='white')
                batch_images.append(img)

            try:
                inputs = self.processor(images=batch_images, return_tensors="pt", padding=True)
                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                with torch.no_grad():
                    image_features = self.model.get_image_features(**inputs)

                # Normalize embeddings
                image_features = image_features / image_features.norm(dim=-1, keepdim=True)
                embeddings.append(image_features.cpu().numpy())
            except Exception as e:
                print(f"Batch error: {e}")
                embeddings.append(np.zeros((len(batch_images), self.image_embed_dim)))

        return np.vstack(embeddings)

    def get_text_embeddings(self, texts):
        embeddings = []

        for i in tqdm(range(0, len(texts), self.batch_size), desc="Processing text"):
            batch_texts = texts[i:i+self.batch_size]

            batch_texts = [str(text)[:500] for text in batch_texts]

            try:
                inputs = self.processor(text=batch_texts, return_tensors="pt",
                                      padding=True, truncation=True)
                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                with torch.no_grad():
                    text_features = self.model.get_text_features(**inputs)

                text_features = text_features / text_features.norm(dim=-1, keepdim=True)
                embeddings.append(text_features.cpu().numpy())
            except Exception as e:
                print(f"Batch error: {e}")
                embeddings.append(np.zeros((len(batch_texts), self.text_embed_dim)))

        return np.vstack(embeddings)

clip_extractor = CLIPFeatureExtractor()

Loading CLIP model: openai/clip-vit-base-patch32
Model loaded successfully!
Embedding dimension: 512
Model loaded successfully!
Embedding dimension: 512


In [ ]:
clip_embeddings_exist = (
    os.path.exists('train_clip_text_embeddings.npy') and 
    os.path.exists('test_clip_text_embeddings.npy') and
    os.path.exists('train_clip_image_embeddings.npy') and 
    os.path.exists('test_clip_image_embeddings.npy')
)

if clip_embeddings_exist:
    print("Loading pre-computed CLIP embeddings...")
    train_text_embeddings = np.load('train_clip_text_embeddings.npy')
    test_text_embeddings = np.load('test_clip_text_embeddings.npy')
    train_image_embeddings = np.load('train_clip_image_embeddings.npy')
    test_image_embeddings = np.load('test_clip_image_embeddings.npy')
else:
    train_text_embeddings = clip_extractor.get_text_embeddings(train_df['catalog_content'].tolist())
    test_text_embeddings = clip_extractor.get_text_embeddings(test_df['catalog_content'].tolist())

    print(f"Train text embeddings shape: {train_text_embeddings.shape}")
    print(f"Test text embeddings shape: {test_text_embeddings.shape}")

    train_image_embeddings = clip_extractor.get_image_embeddings(train_df['image_link'].tolist())
    test_image_embeddings = clip_extractor.get_image_embeddings(test_df['image_link'].tolist())

    print(f"Train image embeddings shape: {train_image_embeddings.shape}")
    print(f"Test image embeddings shape: {test_image_embeddings.shape}")

    np.save('train_clip_text_embeddings.npy', train_text_embeddings)
    np.save('test_clip_text_embeddings.npy', test_text_embeddings)
    np.save('train_clip_image_embeddings.npy', train_image_embeddings)
    np.save('test_clip_image_embeddings.npy', test_image_embeddings)

Processing text:   0%|          | 6/2344 [00:15<1:43:41,  2.66s/it]



Processing text:   0%|          | 6/2344 [00:15<1:43:41,  2.66s/it]



KeyboardInterrupt: 

In [ ]:
def prepare_lightgbm_features():
    X_train_combined = np.hstack([
        train_text_embeddings,  # CLIP text features
        train_image_embeddings,  # CLIP image features
        train_text_df.values  # Engineered numerical features
    ])

    X_test_combined = np.hstack([
        test_text_embeddings,
        test_image_embeddings,
        test_text_df.values
    ])

    print(f"Combined feature shape: {X_train_combined.shape}")

    if Config.USE_SVD and X_train_combined.shape[1] > Config.N_COMPONENTS:
        svd = TruncatedSVD(n_components=Config.N_COMPONENTS, random_state=42)
        X_train_reduced = svd.fit_transform(X_train_combined)
        X_test_reduced = svd.transform(X_test_combined)

    else:
        X_train_reduced = X_train_combined
        X_test_reduced = X_test_combined

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_reduced)
    X_test_scaled = scaler.transform(X_test_reduced)

    print(f"Final training features shape: {X_train_scaled.shape}")
    print(f"Final test features shape: {X_test_scaled.shape}")
    
    return X_train_scaled, X_test_scaled, scaler

X_train_lgb, X_test_lgb, lgb_scaler = prepare_lightgbm_features()

In [ ]:
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denominator
    return np.mean(diff) * 100

def train_lightgbm_model(X_train, y_train, X_test):
    
    n_folds = 5
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    oof_predictions = np.zeros(len(X_train))
    test_predictions = np.zeros(len(X_test))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
        print(f"\nFold {fold} / {n_folds}")

        X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
        y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]

        train_data = lgb.Dataset(X_fold_train, label=y_fold_train)
        val_data = lgb.Dataset(X_fold_val, label=y_fold_val, reference=train_data)

        model = lgb.train(
            Config.LGB_PARAMS,
            train_data,
            num_boost_round=1000,
            valid_sets=[train_data, val_data],
            valid_names=['train', 'valid'],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=100)
            ]
        )

        oof_predictions[val_idx] = model.predict(X_fold_val)
        test_predictions += model.predict(X_test) / n_folds

        fold_smape = smape(y_fold_val, oof_predictions[val_idx])
        fold_scores.append(fold_smape)
        print(f"Fold {fold} SMAPE: {fold_smape:.4f}%")

    overall_smape = smape(y_train, oof_predictions)
    print(f"\n{'='*50}")
    print(colored(f"LightGBM Overall SMAPE: {overall_smape:.4f}%", 'green'))
    print(f"Mean Fold SMAPE: {np.mean(fold_scores):.4f}% (+/- {np.std(fold_scores):.4f}%)")
    print(f"{'='*50}")
    
    return test_predictions, overall_smape

y_train = train_df['price'].values
lgb_predictions, lgb_smape = train_lightgbm_model(X_train_lgb, y_train, X_test_lgb)

In [ ]:
class ProductDataset(Dataset):
    def __init__(self, df, tokenizer, image_embeddings, is_test=False, use_log_price=True):
        self.df = df
        self.tokenizer = tokenizer
        self.image_embeddings = image_embeddings
        self.is_test = is_test
        self.use_log_price = use_log_price

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['catalog_content'])
        image_embedding = self.image_embeddings[idx]

        text_inputs = self.tokenizer(
            text,
            max_length=Config.MAX_TEXT_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        item = {
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0),
            'image_embedding': torch.tensor(image_embedding, dtype=torch.float)
        }

        if not self.is_test:
            price = np.log1p(row['price']) if self.use_log_price else row['price']
            item['price'] = torch.tensor(price, dtype=torch.float)
        return item

In [ ]:
class MultiModalPricer(nn.Module):
    def __init__(self, image_embedding_dim, dropout_rate=0.2):
        super().__init__()
        self.text_tower = DistilBertModel.from_pretrained(Config.TEXT_MODEL)
        combined_dim = self.text_tower.config.dim + image_embedding_dim

        self.regressor = nn.Sequential(
            nn.LayerNorm(combined_dim),
            nn.Linear(combined_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask, image_embedding):
        text_output = self.text_tower(input_ids=input_ids, attention_mask=attention_mask)
        text_embed = text_output.last_hidden_state[:, 0, :]
        combined_features = torch.cat([text_embed, image_embedding], dim=1)
        price_pred = self.regressor(combined_features)
        return price_pred

In [ ]:
def smape_loss(y_pred, y_true):
    numerator = torch.abs(y_pred - y_true)
    denominator = (torch.abs(y_true) + torch.abs(y_pred)).clamp(min=1e-8) / 2
    return torch.mean(numerator / denominator) * 100

def train_neural_network():
    print("\n=== Training Neural Network Model ===")
    
    use_precomputed = (
        os.path.exists(Config.TRAIN_EMBEDDINGS_FILE) and 
        os.path.exists(Config.TEST_EMBEDDINGS_FILE)
    )
    
    if use_precomputed:
        with np.load(Config.TRAIN_EMBEDDINGS_FILE) as data:
            nn_train_embeddings = data['embeddings']
        with np.load(Config.TEST_EMBEDDINGS_FILE) as data:
            nn_test_embeddings = data['embeddings']
    else:
        nn_train_embeddings = train_image_embeddings
        nn_test_embeddings = test_image_embeddings
    
    train_split_df, val_split_df, train_split_embeddings, val_split_embeddings = train_test_split(
        train_df, nn_train_embeddings, test_size=Config.VAL_SPLIT_SIZE, random_state=42
    )
    
    tokenizer = DistilBertTokenizer.from_pretrained(Config.TEXT_MODEL)
    
    train_dataset = ProductDataset(train_split_df, tokenizer, train_split_embeddings, use_log_price=True)
    val_dataset = ProductDataset(val_split_df, tokenizer, val_split_embeddings, use_log_price=True)
    test_dataset = ProductDataset(test_df, tokenizer, nn_test_embeddings, is_test=True)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    image_embedding_dim = nn_train_embeddings.shape[1]
    model = MultiModalPricer(image_embedding_dim=image_embedding_dim).to(device)
    optimizer = AdamW(model.parameters(), lr=Config.LEARNING_RATE)
    loss_fn = nn.MSELoss()
    
    
    best_val_smape = float('inf')
    
    for epoch in range(Config.EPOCHS):
        model.train()
        total_train_loss, total_train_smape = 0, 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{Config.EPOCHS} [Training]")

        for batch in progress_bar:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image_embedding = batch['image_embedding'].to(device)
            prices = batch['price'].to(device)

            predictions = model(input_ids, attention_mask, image_embedding).squeeze()
            loss = loss_fn(predictions, prices)
            loss.backward()
            optimizer.step()

            smape_val = smape_loss(torch.expm1(predictions), torch.expm1(prices))
            total_train_loss += loss.item()
            total_train_smape += smape_val.item()
            progress_bar.set_postfix({
                'Train MSE': f"{loss.item():.4f}",
                'Train SMAPE': f"{smape_val.item():.2f}"
            })

        avg_train_loss = total_train_loss / len(train_loader)
        avg_train_smape = total_train_smape / len(train_loader)
        
        model.eval()
        total_val_loss, total_val_smape = 0, 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch + 1}/{Config.EPOCHS} [Validation]"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                image_embedding = batch['image_embedding'].to(device)
                prices = batch['price'].to(device)

                predictions = model(input_ids, attention_mask, image_embedding).squeeze()
                val_loss = loss_fn(predictions, prices)
                val_smape = smape_loss(torch.expm1(predictions), torch.expm1(prices))

                total_val_loss += val_loss.item()
                total_val_smape += val_smape.item()

        avg_val_loss = total_val_loss / len(val_loader)
        avg_val_smape = total_val_smape / len(val_loader)
        
        
        if avg_val_smape < best_val_smape:
            best_val_smape = avg_val_smape
            torch.save(model.state_dict(), 'best_multimodal_model.pth')
    
    
    model.load_state_dict(torch.load('best_multimodal_model.pth'))
    
    all_predictions = []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image_embedding = batch['image_embedding'].to(device)

            predictions = model(input_ids, attention_mask, image_embedding)
            preds = torch.expm1(predictions.squeeze())  # reverse log1p
            all_predictions.extend(preds.cpu().numpy())

    nn_predictions = np.maximum(np.array(all_predictions), 0.01)
    
    return nn_predictions, best_val_smape

nn_predictions, nn_smape = train_neural_network()

In [ ]:


print("Training LightGBM model for feature importance analysis...")
final_lgb_model = lgb.train(
    Config.LGB_PARAMS,
    lgb.Dataset(X_train_lgb, label=y_train),
    num_boost_round=500,
    verbose_eval=False
)

importance = final_lgb_model.feature_importance(importance_type='gain')

clip_text_features = [f'CLIP_Text_{i}' for i in range(train_text_embeddings.shape[1])]
clip_image_features = [f'CLIP_Image_{i}' for i in range(train_image_embeddings.shape[1])]
engineered_features = train_text_df.columns.tolist()

all_feature_names = clip_text_features + clip_image_features + engineered_features

if Config.USE_SVD:
    feature_names = [f'SVD_Component_{i}' for i in range(X_train_lgb.shape[1])]
else:
    feature_names = all_feature_names[:X_train_lgb.shape[1]]

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

print(colored("\n📊 Top 20 Most Important Features:", 'green'))
print(importance_df.head(20))

final_lgb_model.save_model('final_lightgbm_model.txt')

In [ ]:
lgb_weight = 1 / (lgb_smape + 1e-8)
nn_weight = 1 / (nn_smape + 1e-8)
total_weight = lgb_weight + nn_weight

lgb_weight_norm = lgb_weight / total_weight
nn_weight_norm = nn_weight / total_weight


ensemble_predictions = (lgb_weight_norm * lgb_predictions + 
                       nn_weight_norm * nn_predictions)

ensemble_predictions = np.maximum(ensemble_predictions, 0.01)


In [ ]:
final_submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': ensemble_predictions
})

final_submission_df.to_csv('test_output.csv', index=False)

if final_submission_df['price'].isna().any():
    print("️ NaN values detected!")
elif (final_submission_df['price'] <= 0).any():
    print(" Non-positive prices detected!")
else:
    print("All validations passed!")

print(colored(f"\n Weighted Ensemble Output: test_output.csv", 'green'))
print(colored(f"Combines LightGBM (5-fold CV) + Neural Network (MSE+SMAPE)", 'yellow'))
print(f"\nEnsemble Prediction Statistics:")
print(final_submission_df['price'].describe())
print(f"\nSample predictions:")
print(final_submission_df.head(10))